# DcL-BD Smoke Test — Task 0: Generate a backdoored checkpoint

This notebook runs **Task 0** from `SMOKE_TEST_BRIEF.md`: train a clean CIFAR-10 ConvNet, then run the DcL-BD attack pipeline (Stage 0 → 1 → 2) using `cl_id=0` (PyTorch Inductor) on a single GPU.

**Prerequisites**
- Runtime: GPU (T4 is fine; **Runtime → Change runtime type → T4 GPU**).
- The repo's `src/dlcl.py` and `src/abst_cl_model.py` must have the lazy-TVM patches (so we don't need to install Apache TVM just to use `torch.compile`). Cell 2 below applies the patches idempotently — safe to run on a fresh clone of `main`.

**What this produces** (after the attack finishes):

| Path | What's in it |
|---|---|
| `model_weight/convnet::::cifar10_best.pth` | Clean ConvNet state_dict (M's weights pre-attack). |
| `general_dir/convnet::::cifar10.step0` | Pickled `bd_trigger` from Stage 0 (trigger optimization). |
| `work_dir/convnet::::cifar10::::CL___0::::_GPU_/convnet::::cifar10::::CL___0::::_GPU_.step1` | Pickled `[D, act, tuned_model, bd_trigger]` from Stage 1 (V-search). |
| `work_dir/convnet::::cifar10::::CL___0::::_GPU_/best.tar` | **Main artifact for the smoke test:** pickled `[bd_trigger, MyModel, acc_tuple]` (best epoch by score). |
| `work_dir/convnet::::cifar10::::CL___0::::_GPU_/<epoch>.tar` | Per-epoch snapshots (same payload). |
| `work_dir/convnet::::cifar10::::CL___0::::_GPU_/convnet::::cifar10::::CL___0::::_GPU_.step2` | The final `MyModel` (no `bd_trigger` or acc bundled). |

The smoke test's Tasks 1–3 will load `best.tar` (it has both the model and the trigger).

**Verification metrics** (printed at the end by `evaluate_model`):
- `acc_cl_D_cl` — clean accuracy of M on clean inputs (should be normal, e.g. >85% for CIFAR-10 ConvNet).
- `acc_bd_C_bd` — attack success rate: compiled model C predicts the target label on triggered inputs (should be high, >90%).
- `acc_bd_D_cl` — M's accuracy on triggered inputs (should be high — M is supposed to ignore the trigger).

## 1. Clone the repository

If you've already pushed the patched branch (`claude/competent-mendel-ab57fc`), point `--branch` at it; otherwise clone `main` and rely on Cell 2 to apply the lazy-TVM patches in-place.

In [ ]:
%cd /content
![ -d DLCompilerAttack ] || git clone https://github.com/zimuq/DLCompilerAttack.git
%cd /content/DLCompilerAttack
!git log --oneline -5

## 2. Apply lazy-TVM patches (idempotent)

Without these patches, `import src.dlcl` fails because TVM isn't installed. The cell below rewrites the two files only if the sentinel `_HAS_TVM` isn't already there, so it's safe to re-run.

In [ ]:
import re, pathlib

def patch_dlcl(path):
    src = pathlib.Path(path).read_text()
    if '_HAS_TVM' in src:
        print(f'{path}: already patched')
        return

    # 1) Replace the eager TVM/onnxruntime/torch.export import block.
    old_imports = (
        'import os\n'
        'import time\n\n'
        'import tvm\n'
        'from tvm import relay\n'
        'import json\n'
        'import subprocess\n'
        'import torch\n'
        'from torch.export import export\n'
        'import onnxruntime as ort\n'
        'from pathlib import Path\n'
        'from tvm.contrib import graph_runtime\n'
        'from tvm.runtime import load_module, load_static_library, executor\n'
        'from tvm.contrib import utils, graph_executor\n'
        'import numpy as np\n'
    )
    new_imports = (
        'import os\n'
        'import time\n\n'
        'import json\n'
        'import subprocess\n'
        'import torch\n'
        'from pathlib import Path\n'
        'import numpy as np\n\n'
        'try:\n'
        '    import tvm\n'
        '    from tvm import relay\n'
        '    from tvm.contrib import graph_runtime\n'
        '    from tvm.runtime import load_module, load_static_library, executor\n'
        '    from tvm.contrib import utils, graph_executor\n'
        '    _HAS_TVM = True\n'
        'except ImportError:\n'
        '    tvm = None\n'
        '    relay = None\n'
        '    graph_runtime = None\n'
        '    load_module = load_static_library = executor = None\n'
        '    utils = graph_executor = None\n'
        '    _HAS_TVM = False\n\n'
        'try:\n'
        '    import onnxruntime as ort\n'
        'except ImportError:\n'
        '    ort = None\n\n'
        'try:\n'
        '    from torch.export import export\n'
        'except ImportError:\n'
        '    export = None\n'
    )
    assert old_imports in src, f'{path}: import block not found — was it manually edited?'
    src = src.replace(old_imports, new_imports)

    # 2) Wrap the @tvm.instrument.pass_instrument decorator behind _HAS_TVM.
    old_printir = (
        '@tvm.instrument.pass_instrument\n'
        'class PrintIR:\n'
        '    """Print the name of the pass, the IR, only before passes execute."""\n\n'
        '    def run_before_pass(self, mod, info):\n'
        '        pass\n'
        '        # print("Running pass: {}", info)\n'
        '        # print(mod)\n'
    )
    new_printir = (
        'if _HAS_TVM:\n'
        '    @tvm.instrument.pass_instrument\n'
        '    class PrintIR:\n'
        '        """Print the name of the pass, the IR, only before passes execute."""\n\n'
        '        def run_before_pass(self, mod, info):\n'
        '            pass\n'
        '            # print("Running pass: {}", info)\n'
        '            # print(mod)\n'
        'else:\n'
        '    class PrintIR:  # no-op stand-in when TVM is unavailable\n'
        '        def run_before_pass(self, mod, info):\n'
        '            pass\n'
    )
    assert old_printir in src, f'{path}: PrintIR block not found'
    src = src.replace(old_printir, new_printir)

    # 3) Make TargetDevice tvm fields conditional.
    src = src.replace('self.tvm_dev = tvm.cpu()', 'self.tvm_dev = tvm.cpu() if _HAS_TVM else None')
    src = src.replace('self.tvm_target = tvm.target.Target("llvm")', 'self.tvm_target = tvm.target.Target("llvm") if _HAS_TVM else None')
    src = src.replace('self.tvm_target = tvm.target.cuda(arch="sm_86")', 'self.tvm_target = tvm.target.cuda(arch="sm_86") if _HAS_TVM else None')
    src = src.replace('self.tvm_dev = tvm.cuda()', 'self.tvm_dev = tvm.cuda() if _HAS_TVM else None')

    # 4) Guard tvm_compile and onnx_compile entry points so they raise a clear error.
    src = src.replace(
        '    def tvm_compile(self, model: TorchModel):\n        opt_level = 3\n        tvm_mod, params = relay.frontend.from_onnx(model.onnx_model)',
        '    def tvm_compile(self, model: TorchModel):\n        if not _HAS_TVM:\n            raise RuntimeError(\n                "TVM is not installed; install apache-tvm to use cl_id=1 (TVM compile)."\n            )\n        opt_level = 3\n        tvm_mod, params = relay.frontend.from_onnx(model.onnx_model)'
    )
    src = src.replace(
        '    def onnx_compile(self, model: TorchModel) -> OnnxCompiledModel:\n        target = model.target_device',
        '    def onnx_compile(self, model: TorchModel) -> OnnxCompiledModel:\n        if ort is None:\n            raise RuntimeError(\n                "onnxruntime is not installed; install onnxruntime (or onnxruntime-gpu) to use cl_id=2."\n            )\n        target = model.target_device'
    )
    pathlib.Path(path).write_text(src)
    print(f'{path}: patched')

def patch_abst_cl(path):
    src = pathlib.Path(path).read_text()
    if '_HAS_TVM' in src:
        print(f'{path}: already patched')
        return
    old_imports = (
        'import os\n'
        'import tvm\n'
        'import os.path\n'
        'from typing import List\n'
        'import numpy as np\n'
        'import torch.nn as nn\n'
        'import copy\n'
        'import torch\n'
        'import onnx\n'
        'import json\n'
        'from tvm import relay\n'
    )
    new_imports = (
        'import os\n'
        'import os.path\n'
        'from typing import List\n'
        'import numpy as np\n'
        'import torch.nn as nn\n'
        'import copy\n'
        'import torch\n'
        'import onnx\n'
        'import json\n\n'
        'try:\n'
        '    import tvm\n'
        '    from tvm import relay\n'
        '    _HAS_TVM = True\n'
        'except ImportError:\n'
        '    tvm = None\n'
        '    relay = None\n'
        '    _HAS_TVM = False\n'
    )
    assert old_imports in src, f'{path}: import block not found'
    src = src.replace(old_imports, new_imports)

    old_tvm_init = (
        'class TVMCompiledModel(CompiledModel):\n'
        '    def __init__(self, ori_model: TorchModel, compiled_model):\n'
        '        super().__init__(ori_model, compiled_model)\n'
    )
    new_tvm_init = (
        'class TVMCompiledModel(CompiledModel):\n'
        '    def __init__(self, ori_model: TorchModel, compiled_model):\n'
        '        if not _HAS_TVM:\n'
        '            raise RuntimeError(\n'
        '                "TVM is not installed; TVMCompiledModel requires apache-tvm."\n'
        '            )\n'
        '        super().__init__(ori_model, compiled_model)\n'
    )
    assert old_tvm_init in src, f'{path}: TVMCompiledModel.__init__ not found'
    src = src.replace(old_tvm_init, new_tvm_init)
    pathlib.Path(path).write_text(src)
    print(f'{path}: patched')

patch_dlcl('src/dlcl.py')
patch_abst_cl('src/abst_cl_model.py')

## 3. Install dependencies

Colab already ships PyTorch with CUDA, scipy, sklearn, tqdm, and numpy. The extras the attack needs are HuggingFace `datasets`, `onnx`, and `onnxscript` (recent torch's `torch.onnx.export` requires onnxscript at runtime — without it the V-search stage crashes inside `TorchModel.__init__`).

In [ ]:
!pip install -q --upgrade datasets onnx onnxscript

## 4. GPU sanity check

If this prints `cuda` and a non-empty device name, you're set. Otherwise switch the runtime to GPU.

In [ ]:
import torch
print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))

## 5. Smoke-import check

Make sure the patched modules load before we kick off training.

In [ ]:
import sys
sys.path.insert(0, '/content/DLCompilerAttack')
from src.dlcl import DLCompiler, TargetDevice, _HAS_TVM
from src.attack import load_DLCL
from src import ConvNet
print('Imports OK; TVM available =', _HAS_TVM)

## 6. Train the clean ConvNet (M)

This runs `train_model_clean.py --task_id 0` for 100 epochs of CIFAR-10. Expect ~25–45 min on a T4.

**Output:** `model_weight/convnet::::cifar10_best.pth` (best test-acc checkpoint). The file is overwritten only when test acc beats the prior best.

Tip: if you want to shorten training for a faster smoke test, edit `train_model_clean.py:36` (`num_epochs = 100`). The brief asks for >85% clean acc; CIFAR-10 ConvNet hits that around epoch 30–40.

In [ ]:
%cd /content/DLCompilerAttack
!python train_model_clean.py --task_id 0

Verify the clean checkpoint exists and loads.

In [ ]:
import os
ckpt = 'model_weight/convnet::::cifar10_best.pth'
print('exists:', os.path.isfile(ckpt), '| size (MB):', round(os.path.getsize(ckpt) / 1e6, 2))

## 7. Run the DcL-BD attack (Stage 0 → 1 → 2)

`main.py --task_id 0 --cl_id 0 --hardware_id 0` runs the full attack:
- **Stage 0** (`tri_opt.py`) — 10 epochs of trigger optimization. Saves `general_dir/convnet::::cifar10.step0`.
- **Stage 1** (`v_search.py`) — Channel V-search using compiled-vs-pre-compile embedding gaps. Saves `work_dir/<task>/<task>.step1`.
- **Stage 2** (`finetune.py`) — 50 epochs of fine-tuning the tuned half of the model. Evaluates every 10 epochs (and at epoch 0). Saves `<epoch>.tar` and tracks `best.tar` by `3*acc_bd_C_bd + acc_cl_D_cl + acc_bd_D_cl`.

Total wall-clock: roughly 1–2h on a T4. Watch `acc_bd_C_bd` climb in the printed metrics — that's the attack succeeding.

In [ ]:
%cd /content/DLCompilerAttack
!python main.py --task_id 0 --cl_id 0 --hardware_id 0

## 8. Inspect produced artifacts

In [ ]:
!ls -la model_weight/ general_dir/
!echo '---'
!ls -la 'work_dir/convnet::::cifar10::::CL___0::::_GPU_/'

## 9. Verify the checkpoint passes the brief's acceptance bar

Reload `best.tar`, recompute the five accuracy metrics on the test loader, and check:
- `acc_cl_D_cl` (clean acc on M) > 0.85 → M is benign and competent.
- `acc_bd_C_bd` (compiled C on triggered → target label) > 0.90 → compilation flips the behavior (attack lands).
- `acc_bd_D_cl` (M on triggered → ground-truth label) > 0.85 → M ignores the trigger pre-compile.

These are exactly the three numbers `evaluate_model` printed at the end of the attack run, so this cell is mostly a sanity-check that `best.tar` is loadable from a fresh interpreter (which is how Tasks 1–3 of the smoke test will use it).

In [ ]:
import torch, sys
sys.path.insert(0, '/content/DLCompilerAttack')
from src.attack.utils import evaluate_model, CLSetting
from src.attack import load_DLCL
from src import TargetDevice
from utils import load_dataloader

WORK = 'work_dir/convnet::::cifar10::::CL___0::::_GPU_'
best_path = f'{WORK}/best.tar'
bd_trigger, model, saved_acc = torch.load(best_path, weights_only=False)
print('Saved acc tuple (acc_cl_D_cl, acc_cl_C_cl, acc_bd_D_cl, acc_bd_D_bd, acc_bd_C_bd):')
for n, a in zip(['acc_cl_D_cl','acc_cl_C_cl','acc_bd_D_cl','acc_bd_D_bd','acc_bd_C_bd'], saved_acc):
    print(f'  {n:14s} = {float(a):.4f}')

# Brief's acceptance bar
acc_cl_D_cl = float(saved_acc[0])
acc_bd_D_cl = float(saved_acc[2])
acc_bd_C_bd = float(saved_acc[4])
ok = acc_cl_D_cl > 0.85 and acc_bd_C_bd > 0.90 and acc_bd_D_cl > 0.85
print('\nPASS' if ok else '\nDOES NOT MEET BAR — reconsider hyperparams / re-run Stage 2 longer')

## 10. (Optional) Persist artifacts to Drive

Colab disks are ephemeral. If you want the `best.tar` and clean checkpoint to survive a runtime restart, copy them to Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/dclbd_smoke/work_dir
!cp -r 'work_dir/convnet::::cifar10::::CL___0::::_GPU_' /content/drive/MyDrive/dclbd_smoke/work_dir/
!cp -r model_weight /content/drive/MyDrive/dclbd_smoke/
!cp -r general_dir /content/drive/MyDrive/dclbd_smoke/
!ls /content/drive/MyDrive/dclbd_smoke/

---

**Done with Task 0.** The next step (Tasks 1–3 of the smoke test) is the single-instance inconsistency check + Inductor-flag ablation loop. Those run on **CPU** (per the brief), so we'll write them in a separate notebook/script that just loads `best.tar`.